In [ ]:
import pandas as pd
from io import StringIO

# read text file and only keep trip records
with open("trips.txt", "r", encoding="cp1252", errors="replace") as f:
    t_lines = [line for line in f if line.startswith("T;")]

# parse the trip records as pandas df
df_trips = pd.read_csv(StringIO("".join(t_lines)), sep=";", header=None, engine="python")

# select only columns with useful information
df_trips = df_trips.iloc[:, [0,1,2,5,6,7,8,-3]]

df_trips.columns = [
    "record_type",
    "line_number",
    "trip_number",
    # "trip_direction",
    # "variant_code",
    "from_stop",
    "start_time_min",
    "end_time_min",
    "to_stop",
    "distance_km"
]

df_trips["distance_km"] = pd.to_numeric(df_trips["distance_km"], errors="coerce")
df_trips.head()

,record_type,line_number,trip_number,from_stop,start_time_min,end_time_min,to_stop,distance_km
0,T,u001,1001,utrvec,350,389,utrijs,12.983
1,T,u001,1003,utrvec,364,404,utrijs,12.983
2,T,u001,1005,utrvec,377,419,utrijs,12.983
3,T,u001,1007,utrvec,389,434,utrijs,12.983
4,T,u001,1009,utrvec,403,449,utrijs,12.983


In [3]:
# read text file and only keep deadhead records

with open("dhd.txt", "r", encoding="cp1252", errors="replace") as f:
    t_lines = [line for line in f if line.startswith("D;")]

# parse the deadhead records as pandas df
df_deadheads = pd.read_csv(StringIO("".join(t_lines)), sep=";", header=None, engine="python")

# select only columns with useful information
df_deadheads = df_deadheads.iloc[:, [0,1,2,3,4,5,6]]

df_deadheads.columns = [
    "record_type",
    "from_stop-to_stop",
    "time_ver_0",
    "time_ver_1",
    "time_ver_2",
    "time_ver_3",
    "distance_km",
]
df_deadheads["distance_km"] = pd.to_numeric(df_deadheads["distance_km"], errors="coerce")
df_deadheads[["from_stop", "to_stop"]] = df_deadheads["from_stop-to_stop"].str.split("-", n=1, expand=True)
df_deadheads.head()

,record_type,from_stop-to_stop,time_ver_0,time_ver_1,time_ver_2,time_ver_3,distance_km,from_stop,to_stop
0,D,amdjwp-nwggam,26,26,26,26,20.0,amdjwp,nwggam
1,D,amdpri-nwggam,27,27,27,27,20.9,amdpri,nwggam
2,D,amdpri-sllgar,25,34,25,25,19.0,amdpri,sllgar
3,D,amdpri-utrgar,40,60,40,40,28.0,amdpri,utrgar
4,D,amdpri-wbdgar,55,65,55,55,43.0,amdpri,wbdgar


In [5]:
with open("parameters.txt", "r", encoding="cp1252", errors="replace") as f:
    t_lines = [line for line in f if line.startswith("U;")]

# read text file and only keep parameters records
df_params = pd.read_csv(StringIO("".join(t_lines)), sep=";", header=None, engine="python")

# parse the parameters records as pandas df
df_params = df_params.iloc[:, [0,2,4,9,10,12]]

# select only columns with useful information
df_params.columns = [
    "record_type",
    "cost_per_bus",
    # "cost_per_minute",
    "cost_per_km",
    "energy_compsumtion_kwh/km",
    "max_charge_rate_kwh/min",
    "battery_capacity_kwh"
]

df_params.head()

,record_type,cost_per_bus,cost_per_km,energy_compsumtion_kwh/km,max_charge_rate_kwh/min,battery_capacity_kwh
0,U,244.13,0.13,1.3,2.5,199.04


In [4]:
def feasible_arcs(trips, deadheads, max_wait = 10):
    arcs = []

    for i in trips.itertuples():
        for j in trips.itertuples():
            if i.trip_number == j.trip_number:
                continue
            
            if i.to_stop == j.from_stop:
                travel_time = 0
            else:

                dh = deadheads[(deadheads.from_stop == i.to_stop) & (deadheads.to_stop == j.from_stop)]

                if dh.empty:
                    continue

                travel_time = dh.iloc[0]["time_ver_0"]

            slack = j.start_time_min - (i.end_time_min + travel_time)

            if 0 <= slack <= max_wait:
                arcs.append((i.trip_number, j.trip_number))

    return arcs
        
feasible_arcs(df_trips, df_deadheads)

[(1001, 1006),
 (1003, 1008),
 (1005, 1010),
 (1007, 1014),
 (1009, 1016),
 (1011, 1018),
 (1013, 1020),
 (1015, 1022),
 (1017, 1024),
 (1019, 1026),
 (1021, 1028),
 (1023, 1030),
 (1025, 1032),
 (1027, 1034),
 (1029, 1036),
 (1031, 1038),
 (1033, 1040),
 (1035, 1042),
 (1037, 1044),
 (1039, 1046),
 (1041, 1048),
 (1043, 1050),
 (1045, 1052),
 (1047, 1054),
 (1049, 1056),
 (1051, 1058),
 (1053, 1060),
 (1055, 1062),
 (1057, 1064),
 (1059, 1066),
 (1061, 1068),
 (1063, 1070),
 (1065, 1072),
 (1067, 1074),
 (1069, 1076),
 (1071, 1078),
 (1073, 1080),
 (1075, 1082),
 (1077, 1084),
 (1079, 1086),
 (1081, 1088),
 (1083, 1090),
 (1085, 1092),
 (1087, 1094),
 (1089, 1096),
 (1091, 1098),
 (1093, 1100),
 (1095, 1102),
 (1097, 1104),
 (1099, 1106),
 (1101, 1108),
 (1103, 1110),
 (1105, 1112),
 (1107, 1114),
 (1109, 1116),
 (1111, 1118),
 (1113, 1120),
 (1115, 1122),
 (1117, 1124),
 (1119, 1126),
 (1121, 1128),
 (1123, 1130),
 (1125, 1132),
 (1127, 1134),
 (1129, 1136),
 (1131, 1138),
 (1133, 11